# 02 Crazyflie Multi-Simulation

Runs a batch of Crazyflie simulations with different configurations (noise levels, flight duration).
Each run is exported to its own subdirectory under `simulated_data/02_crazyflie_multi/`.

In [ ]:
from dataclasses import dataclass
from dataclasses import replace

import numpy as np
from matplotlib import pyplot as plt

from pipeline.dropout import DropoutInjector
from pipeline.noise import NoiseInjector
from pipeline.noise.core.noise_profile import NoiseProfile
from pipeline.synthetic import SyntheticSensorGenerator, SyntheticTimeOfFlightDistance, SyntheticUWBTransceiver
from rotorpy_simulation.crazyflie import Crazyflie, CrazyflieIMU
from rotorpy_simulation.crazyflie_simulation import CrazyflieSimulation
from rotorpy_simulation.crazyflie.trajectories import get_circular_trajectory


In [ ]:
BASE_DATA_PATH: str = "../../simulated_data/rotorpy/"
plt.style.use('default')

In [ ]:
from rotorpy_simulation.crazyflie.trajectories import (
    get_circular_trajectory,
    get_fast_circular_trajectory,
    get_varying_height_circular_trajectory,
    get_figure8_trajectory,
    get_vertical_oscillation_trajectory,
)


@dataclass
class SimulationRun:
    name: str
    trajectory: any = None  # set in __post_init__ to avoid mutable default
    accel_noise_multiplier: float = 1.0
    gyro_noise_multiplier: float = 1.0
    tof_noise_multiplier: float = 1.0
    uwb_noise_multiplier: float = 1.0
    uwb_dropout_probability: float = 0.0
    simulation_duration: float = 15.0  # seconds

    def __post_init__(self):
        if self.trajectory is None:
            self.trajectory = get_circular_trajectory()


## 1. Run Configurations

In [ ]:
CRAZYFLIE_RUNS: list[SimulationRun] = [
    # Low noise
    SimulationRun(
        name="control_circular",
        trajectory=get_circular_trajectory(),
        accel_noise_multiplier=0.5,
        gyro_noise_multiplier=0.5,
        tof_noise_multiplier=0.5,
        uwb_noise_multiplier=0.5,
        uwb_dropout_probability=0.0,
        simulation_duration=15.0,
    ),

    # Varying height circle, tests position and altitude estimation
    SimulationRun(
        name="varying_height_circle",
        trajectory=get_varying_height_circular_trajectory(),
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        tof_noise_multiplier=1.0,
        uwb_noise_multiplier=1.0,
        uwb_dropout_probability=0.0,
        simulation_duration=20.0,
    ),

    # Varying height circle with high ToF noise, evaluates altitude estimation under bad ToF
    SimulationRun(
        name="varying_height_circle_bad_tof",
        trajectory=get_varying_height_circular_trajectory(),
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        tof_noise_multiplier=1.5,
        uwb_noise_multiplier=1.0,
        uwb_dropout_probability=0.0,
        simulation_duration=20.0,
    ),

    # Fast circular, stresses IMU under fast flights and elevated noise
    SimulationRun(
        name="fast_circular_noisy",
        trajectory=get_fast_circular_trajectory(),
        accel_noise_multiplier=1.2,
        gyro_noise_multiplier=1.2,
        tof_noise_multiplier=1.2,
        uwb_noise_multiplier=1.2,
        uwb_dropout_probability=0.1,
        simulation_duration=15.0,
    ),

    # Figure-8 with faulty UWB, evaluates horizontal estimation under bad UWB sensors
    SimulationRun(
        name="figure8_bad_uwb",
        trajectory=get_figure8_trajectory(),
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        tof_noise_multiplier=1.0,
        uwb_noise_multiplier=1.5,
        uwb_dropout_probability=0.5,
        simulation_duration=20.0,
    ),

    # Vertical oscillation, tests vertical estimation using higher IMU and ToF noise
    SimulationRun(
        name="vertical_oscillation_noisy",
        trajectory=get_vertical_oscillation_trajectory(),
        accel_noise_multiplier=1.2,
        gyro_noise_multiplier=1.2,
        tof_noise_multiplier=1.2,
        uwb_noise_multiplier=1.0,
        uwb_dropout_probability=0.0,
        simulation_duration=15.0,
    ),

    # Evaluation Flight
    SimulationRun(
        name="crazyflie_evaluation",
        trajectory=get_circular_trajectory(),
        accel_noise_multiplier=1.0,
        gyro_noise_multiplier=1.0,
        tof_noise_multiplier=1.0,
        uwb_noise_multiplier=1.0,
        uwb_dropout_probability=0.0,
        simulation_duration=20.0,
    ),
]

In [ ]:
for run in CRAZYFLIE_RUNS:
    print(run)

## 2. Setup

In [ ]:
from pipeline.dropout.core.dropout_strategy import RandomDropout

tof_distance_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/VL53L1x_ToF-Distance.yaml")
uwb_transceiver_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/DWM1000_UWB_Transceiver.yaml")


def scale_noise(profile: NoiseProfile, multiplier: float) -> NoiseProfile:
    return replace(
        profile,
        noise_density=profile.noise_density * multiplier,
        random_walk=profile.random_walk * multiplier,
    )


def build_scaled_imu(accel_multiplier: float, gyro_multiplier: float) -> CrazyflieIMU:
    imu = CrazyflieIMU()
    _accel_base = imu.accelerometer_params
    _gyro_base = imu.gyroscope_params
    imu.accelerometer_params = {**_accel_base,
                                'initial_bias': _accel_base['initial_bias'] * accel_multiplier,
                                'noise_density': _accel_base['noise_density'] * accel_multiplier,
                                }
    imu.gyroscope_params = {**_gyro_base,
                            'initial_bias': _gyro_base['initial_bias'] * gyro_multiplier,
                            'noise_density': _gyro_base['noise_density'] * gyro_multiplier,
                            }
    return imu


def build_synthetic_sensors(tof_noise_multiplier: float, uwb_multiplier: float, uwb_dropout_probability: float) -> list:
    dropout = DropoutInjector([RandomDropout(uwb_dropout_probability)]) if uwb_dropout_probability > 0.0 else None
    uwb_p = scale_noise(uwb_transceiver_noise_profile, uwb_multiplier)
    tof_p = scale_noise(tof_distance_noise_profile, tof_noise_multiplier)
    return [
        (SyntheticTimeOfFlightDistance(name="time_of_flight_distance"),
         NoiseInjector(tof_p, channels=[0]), None),
        (SyntheticUWBTransceiver(name="uwb_transceiver_0", anchor_position=np.array([2.32, 1.39, 2.08]), anchor_id=0),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_1", anchor_position=np.array([2.32, -1.43, 2.08]), anchor_id=1),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_2", anchor_position=np.array([2.34, 1.39, 0.23]), anchor_id=2),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_3", anchor_position=np.array([2.32, -1.43, 0.20]), anchor_id=3),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_4", anchor_position=np.array([-2.32, 1.39, 2.25]), anchor_id=4),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_5", anchor_position=np.array([-2.32, -1.43, 1.87]), anchor_id=5),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_6", anchor_position=np.array([-2.32, 1.39, 0.23]), anchor_id=6),
         NoiseInjector(uwb_p, channels=[0]), dropout),
        (SyntheticUWBTransceiver(name="uwb_transceiver_7", anchor_position=np.array([-2.32, -1.43, 0.24]), anchor_id=7),
         NoiseInjector(uwb_p, channels=[0]), dropout),
    ]

## 3. Execute All Runs
For each run: simulate, add synthetic sensors with scaled noise, export.

In [ ]:
results = {}

for run in CRAZYFLIE_RUNS:
    print(f"\n{'=' * 60}")
    print(
        f"Run: {run.name}")
    print(f"{'=' * 60}")

    run_path = BASE_DATA_PATH + run.name

    # --- 1. Object Generation & Run ---
    simulation = CrazyflieSimulation(
        crazyflie_builder=Crazyflie(),
        trajectory=run.trajectory,
        imu_sensor_builder=build_scaled_imu(run.accel_noise_multiplier, run.gyro_noise_multiplier),
        simulation_duration=run.simulation_duration,
    )
    result = simulation.run(run.name)

    # --- 2. Synthetic Sensor Addition ---
    synthetic_sensors = build_synthetic_sensors(run.tof_noise_multiplier, run.uwb_noise_multiplier,
                                                run.uwb_dropout_probability)
    generator = SyntheticSensorGenerator(sensors=synthetic_sensors)
    generator.apply(result)

    # --- 3. Noise Injection ---
    # IMU noise is already baked into the RotorPy simulation via build_scaled_imu.
    # Synthetic sensor noise is applied inside build_synthetic_sensors.

    # --- 4. Dropout Simulation ---
    # directly applied in synthetic sensor generation

    # --- 5. Export ---
    result.export_csv_data(run_path)
    result.save(f"{run_path}/{run.name}.pkl")

    results[run.name] = result

print("\nAll runs complete.")

## 4. Inspect
Quick sanity-check on one result.

In [ ]:
import plotly.graph_objects as go
import plotly.colors

labels = [SimulationRun.name for SimulationRun in CRAZYFLIE_RUNS]
fig = go.Figure()

colors = plotly.colors.qualitative.Plotly

for i, label in enumerate(labels):
    pos = results[label].ground_truth.position  # (N, 3) ENU
    trace_color = colors[i % len(colors)]

    fig.add_trace(go.Scatter3d(
        x=pos[:, 0],
        y=pos[:, 1],
        z=pos[:, 2],
        mode="lines",
        name=label,
        line=dict(width=4, color=trace_color),
        legendgroup=label
    ))

    fig.add_trace(go.Scatter3d(
        x=[pos[0, 0]],
        y=[pos[0, 1]],
        z=[pos[0, 2]],
        mode="markers",
        marker=dict(size=4, color=trace_color, symbol="circle"),
        showlegend=False,
        legendgroup=label,
        hoverinfo="skip"
    ))

# Set labels, title and equal axis scaling
fig.update_layout(
    title="Flight trajectories",
    scene=dict(
        xaxis_title="East [m]",
        yaxis_title="North [m]",
        zaxis_title="Up [m]",
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()